# Capstone Two: Data Wrangling

## Dataset: Online Shoppers Purchasing Intention

## The goal of this notebook is to do a short, practical wrangling pass before EDA and machine-learning preprocessing. I will load the data, inspect its structure, check missing values and duplicates, review data types and ranges, inspect outliers, and save a cleaned copy.

## 1. Load the data

## There is one CSV file, so no joining or merging is needed.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

DATA_PATH = Path("online_shoppers_intention.csv")

if not DATA_PATH.exists():
    DATA_PATH = Path("D:/springboard/case studies/Capstone 2/Data wrangling/data/online_shoppers_intention.csv")

df_raw = pd.read_csv(DATA_PATH)

print("Shape:", df_raw.shape)
df_raw.head()

Shape: (12330, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


## 2. Understand the columns

## The dataset has page-count and duration fields (`Administrative`, `Informational`, `ProductRelated`), session metrics (`BounceRates`, `ExitRates`, `PageValues`, `SpecialDay`), visitor/session categories, and `Revenue` as the purchase target.

## Some columns such as `Browser`, `Region`, and `TrafficType` are stored as integers, but the numbers are category codes rather than measurements.

In [3]:
# Quick structure check
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  object 
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType           

In [5]:
# Compact summary: type, missing values and number of unique values
column_summary = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing": df_raw.isna().sum(),
    "unique": df_raw.nunique()
})

column_summary

,dtype,missing,unique
Administrative,int64,0,27
Administrative_Duration,float64,0,3335
Informational,int64,0,17
Informational_Duration,float64,0,1258
ProductRelated,int64,0,311
ProductRelated_Duration,float64,0,9551
BounceRates,float64,0,1872
ExitRates,float64,0,4777
PageValues,float64,0,2704
SpecialDay,float64,0,6


In [7]:
# Numeric ranges help identify obviously unusual or invalid values
df_raw.select_dtypes(include=np.number).agg(["min", "median", "max"]).T

,min,median,max
Administrative,0.0,1.000000,27.000000
Administrative_Duration,0.0,7.500000,3398.750000
Informational,0.0,0.000000,24.000000
Informational_Duration,0.0,0.000000,2549.375000
ProductRelated,0.0,18.000000,705.000000
ProductRelated_Duration,0.0,598.936905,63973.522230
BounceRates,0.0,0.003112,0.200000
ExitRates,0.0,0.025156,0.200000
PageValues,0.0,0.000000,361.763742
SpecialDay,0.0,0.000000,1.000000


## 3. Check missing values

In [9]:
missing = df_raw.isna().sum()
print("Total missing values:", int(missing.sum()))
missing[missing > 0]

Total missing values: 0


Series([], dtype: int64)

## 4. Check and remove exact duplicates

## Exact duplicate rows can overweight repeated sessions, so I removed them.

In [11]:
duplicate_count = int(df_raw.duplicated().sum())
print("Exact duplicate rows:", duplicate_count)

df = df_raw.drop_duplicates().reset_index(drop=True)

print("Rows before:", len(df_raw))
print("Rows after :", len(df))

Exact duplicate rows: 125
Rows before: 12330
Rows after : 12205


## 5. Clean up data types

## The coded ID columns are categorical even though they are stored as numbers. `Month` and `VisitorType` are also categorical. `Weekend` and `Revenue` are already Boolean.

In [13]:
categorical_columns = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType"
]

for col in categorical_columns:
    df[col] = df[col].astype("category")

df.dtypes

Administrative                int64
Administrative_Duration     float64
Informational                 int64
Informational_Duration      float64
ProductRelated                int64
ProductRelated_Duration     float64
BounceRates                 float64
ExitRates                   float64
PageValues                  float64
SpecialDay                  float64
Month                      category
OperatingSystems           category
Browser                    category
Region                     category
TrafficType                category
VisitorType                category
Weekend                        bool
Revenue                        bool
dtype: object

## 6. Basic validity checks

## Counts and durations should not be negative. I also checked the observed ranges of the rate-like fields.

In [15]:
non_negative_columns = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration"
]

negative_counts = (df[non_negative_columns] < 0).sum()
print("Negative values found:")
print(negative_counts)

print("\nObserved ranges:")
df[["BounceRates", "ExitRates", "SpecialDay"]].agg(["min", "max"])

Negative values found:
Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
dtype: int64

Observed ranges:


,BounceRates,ExitRates,SpecialDay
min,0.0,0.0,0.0
max,0.2,0.2,1.0


## 7. Review possible outliers

## Used the 1.5 × IQR rule only as a flagging method

In [17]:
outlier_columns = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay"
]

outlier_rows = []

for col in outlier_columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    flagged = ((df[col] < lower) | (df[col] > upper)).sum()

    outlier_rows.append({
        "column": col,
        "lower_bound": round(lower, 3),
        "upper_bound": round(upper, 3),
        "flagged_rows": int(flagged)
    })

outlier_summary = pd.DataFrame(outlier_rows)
outlier_summary

,column,lower_bound,upper_bound,flagged_rows
0,Administrative,-6.000,10.000,404
1,Administrative_Duration,-142.050,236.750,1149
2,Informational,0.000,0.000,2631
3,Informational_Duration,0.000,0.000,2405
4,ProductRelated,-37.000,83.000,1007
5,ProductRelated_Duration,-1733.232,3403.387,951
6,BounceRates,-0.025,0.042,1428
7,ExitRates,-0.037,0.100,1325
8,PageValues,0.000,0.000,2730
9,SpecialDay,0.000,0.000,1249


## Several variables have many IQR outliers. This dataset contains strongly skewed browsing behavior and several zero-heavy columns, so the IQR rule can flag valid sessions as outliers. I did not remove or cap these values because I did not find evidence that they are data-entry errors. Transformations can be considered later during preprocessing if a model needs them.

## 8. Final check and save cleaned data

In [21]:
print("Final shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

print("\nRevenue distribution (%):")
print((df["Revenue"].value_counts(normalize=True) * 100).round(2))

OUTPUT_PATH = Path("D:/springboard/case studies/Capstone 2/Data wrangling/data/online_shoppers_intention_cleaned.csv")
df.to_csv(OUTPUT_PATH, index=False)

print("\nSaved:", OUTPUT_PATH)

Final shape: (12205, 18)
Missing values: 0
Duplicate rows: 0

Revenue distribution (%):
Revenue
False    84.37
True     15.63
Name: proportion, dtype: float64

Saved: D:\springboard\case studies\Capstone 2\Data wrangling\data\online_shoppers_intention_cleaned.csv
